<a href="https://colab.research.google.com/github/Lee-yo-han03/Sql-data-warehouse_project/blob/main/SQL_Data_Warehouse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# PostgreSQL 서버 및 추가 기능 설치
!apt-get update -qq
!apt-get install -y postgresql postgresql-contrib

# PostgreSQL 서버 시작
!service postgresql start

# 서버 상태 확인
!pg_lsclusters


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libtypes-serialiser-perl
  logrotate postgresql-16 postgresql-client-16 postgresql-client-common
  postgresql-common ssl-cert sysstat
Suggested packages:
  bsd-mailx | mailx postgresql-doc postgresql-doc-16 isag
The following NEW packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libtypes-serialiser-perl
  logrotate postgresql postgresql-16 postgresql-client-16
  postgresql-client-common postgresql-common postgresql-contrib ssl-cert
  sysstat
0 upgraded, 13 newly installed, 0 to remove and 6 not upgraded.
Need to get 17.9 MB of archives.
After this operation, 52.9 MB of add

In [ ]:

%pip install -q psycopg2-binary

# 연습용 PostgreSQL 계정 비밀번호 설정
!sudo -u postgres psql -c "ALTER USER postgres WITH PASSWORD 'colab_local_only';"

print("Python 연결 라이브러리 및 계정 설정 완료")


ALTER ROLE
Python 연결 라이브러리 및 계정 설정 완료


In [ ]:
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres WITH PASSWORD 'postgres';"

[ OK ]
ALTER ROLE


In [ ]:
import psycopg2

# 1. PostgreSQL 서버 시작
!service postgresql start

# 2. PostgreSQL 접속 설정
DB_CONFIG = {
    "user": "postgres",
    "password": "postgres",
    "host": "127.0.0.1",
    "port": 5432
}

# 3. 기본 데이터베이스에 접속
conn = psycopg2.connect(
    dbname="postgres",
    **DB_CONFIG
)
conn.autocommit = True

print("PostgreSQL 연결 성공!")
print("연결 상태:", conn.closed)  # 0이면 연결 중

# 4. datawarehouse 데이터베이스 확인 및 생성
with conn.cursor() as cur:
    cur.execute("""
        SELECT 1
        FROM pg_database
        WHERE datname = 'datawarehouse';
    """)

    if cur.fetchone() is None:
        cur.execute("CREATE DATABASE datawarehouse;")
        print("datawarehouse 데이터베이스 생성 완료")
    else:
        print("datawarehouse 데이터베이스가 이미 존재합니다.")

conn.close()

# 5. datawarehouse 데이터베이스에 접속
conn = psycopg2.connect(
    dbname="datawarehouse",
    **DB_CONFIG
)

# 6. Bronze, Silver, Gold 스키마 생성
with conn.cursor() as cur:
    cur.execute("CREATE SCHEMA IF NOT EXISTS bronze;")
    cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
    cur.execute("CREATE SCHEMA IF NOT EXISTS gold;")

    cur.execute("""
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name IN ('bronze', 'silver', 'gold')
        ORDER BY schema_name;
    """)

    print("\n생성된 스키마:")
    for row in cur.fetchall():
        print(row[0])

conn.commit()

print("\n데이터 웨어하우스 초기 설정 완료!")
print("Datawarehouse 연결 상태:", conn.closed)

[ OK ]
PostgreSQL 연결 성공!
연결 상태: 0
datawarehouse 데이터베이스 생성 완료

생성된 스키마:
bronze
gold
silver

데이터 웨어하우스 초기 설정 완료!
Datawarehouse 연결 상태: 0


In [ ]:
#Making the bronze layer tables
#The drop key is deleting the existing table, so key IF NOT EXISTS is not necessary. JUST TO STUDY
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""DROP TABLE IF EXISTS bronze.crm_cust_info;""" )
  cur.execute("""CREATE TABLE IF NOT EXISTS bronze.crm_cust_info(
                  cst_id INTEGER,
                  cst_key VARCHAR(50),
                  cst_firstname VARCHAR(50),
                  cst_lastname VARCHAR(50),
                  cst_material_status VARCHAR(50),
                  cst_material_gndr VARCHAR(50),
                  cst_create_date DATE);""")

  cur.execute("""DROP TABLE IF EXISTS bronze.crm_prd_info;
                 CREATE TABLE IF NOT EXISTS bronze.crm_prd_info(
                  prd_id INTEGER,
                  prd_key VARCHAR(50),
                  prd_nm VARCHAR(50),
                  prd_cost INTEGER,
                  prd_line VARCHAR(50),
                  prd_start_dt DATE,
                  prd_end_dt DATE);""")

  cur.execute("""DROP TABLE IF EXISTS bronze.crm_sales_details;
                 CREATE TABLE IF NOT EXISTS bronze.crm_sales_details(
                  sls_ord_num VARCHAR(50),
                  sls_prd_key VARCHAR(50),
                  sls_cust_id INTEGER,
                  sls_order_dt INTEGER,
                  sls_ship_dt INTEGER,
                  sls_due_dt INTEGER,
                  sls_sales INTEGER,
                  sls_quantity INTEGER,
                  sls_price INTEGER);""")

  cur.execute("""DROP TABLE IF EXISTS bronze.erp_cust_az12;
                 CREATE TABLE IF NOT EXISTS bronze.erp_cust_az12(
                  cid VARCHAR(50),
                  bdate DATE,
                  gen VARCHAR(50));""")

  cur.execute("""DROP TABLE IF EXISTS bronze.erp_loc_a101;
                 CREATE TABLE IF NOT EXISTS bronze.erp_loc_a101(
                  cid VARCHAR(50),
                  cntry VARCHAR(50));""")

  cur.execute("""DROP TABLE IF EXISTS  bronze.erp_px_cat_g1v2;
                 CREATE TABLE IF NOT EXISTS bronze.erp_px_cat_g1v2(
                  id VARCHAR(50),
                  cat VARCHAR(50),
                  subcat VARCHAR(50),
                  maintenance VARCHAR(50));""")
  conn.commit()



In [ ]:
#COPY INSTEAD OF BULT INSERT
from google.colab import files
uploaded = files.upload()

Saving sql-data-warehouse-project.zip to sql-data-warehouse-project.zip


In [ ]:
#파일 업로드 되었는지 확인
import os
print(os.listdir('/content'))

['.config', 'sql-data-warehouse-project.zip', 'drive', 'sample_data']


In [ ]:
#ZIP파일 풀기
import zipfile
zip_path = "/content/sql-data-warehouse-project.zip"
extract_path = "/content/sql-data-warehouse-project"
with zipfile.ZipFile(zip_path,"r") as z:
  z.extractall(extract_path)
print(os.listdir(extract_path))

['sql-data-warehouse-project']


In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    if "cust_az12.csv" in files:
        print(os.path.join(root, "cust_az12.csv"))

In [ ]:
#How to make a procedure.(we can include the copy work in here!)
#cur.execute("TRUNCATE TABLE bronze.crm_cust_info;")#If we don't use this code, then the number of row becomes doubled everytime we implement it
#RAISE NOTICE 'ERROR OCCERED%', SQLERRM; #SHOUING THE CONTENT OF THE ERROR
#RAISE; #RAISE THE ERROR SO THAT WE CAN PROCEDE NEXT STEP TO DEAL WITH THE ERROR
conn.rollback()
with conn.cursor() as cur:
    cur.execute("""
        CREATE OR REPLACE PROCEDURE bronze.truncate_bronze()
        LANGUAGE plpgsql
        AS $$
        DECLARE
            start_time TIMESTAMP;
            end_time TIMESTAMP;
        BEGIN
            -- CRM 테이블 초기화 시간 측정
            start_time := clock_timestamp();

            TRUNCATE TABLE
                bronze.crm_cust_info,
                bronze.crm_prd_info,
                bronze.crm_sales_details;

            end_time := clock_timestamp();

            RAISE NOTICE 'CRM tables truncated';
            RAISE NOTICE 'CRM truncate duration: % seconds',
                ROUND(
                    EXTRACT(EPOCH FROM (end_time - start_time))::numeric,
                    3
                );

            -- ERP 테이블 초기화 시간 측정
            start_time := clock_timestamp();

            TRUNCATE TABLE
                bronze.erp_cust_az12,
                bronze.erp_loc_a101,
                bronze.erp_px_cat_g1v2;

            end_time := clock_timestamp();

            RAISE NOTICE 'ERP tables truncated';
            RAISE NOTICE 'ERP truncate duration: % seconds',
                ROUND(
                    EXTRACT(EPOCH FROM (end_time - start_time))::numeric,
                    3
                );

        EXCEPTION
            WHEN OTHERS THEN
                RAISE NOTICE 'Error occurred during truncation';
                RAISE NOTICE 'Error details: %', SQLERRM;
                RAISE;
        END;
        $$;
    """)

conn.commit()
print("Bronze truncate procedure created successfully!")
# 이전 NOTICE 메시지 초기화
conn.notices.clear()

# 프로시저 실행
with conn.cursor() as cur:
    cur.execute("CALL bronze.truncate_bronze();")

conn.commit()
# Colab 출력창에 메시지 표시
for notice in conn.notices:
    print(notice.strip())

Bronze truncate procedure created successfully!
NOTICE:  CRM tables truncated
NOTICE:  CRM truncate duration: 0.001 seconds
NOTICE:  ERP tables truncated
NOTICE:  ERP truncate duration: 0.001 seconds


In [ ]:
#I decided to use Copy instead of using BULK INSERT because there's no bulk insert concept in postgresql
with conn.cursor() as cur:
    cur.execute("""CALL bronze.truncate_bronze();""")
    cur.execute("""
        COPY bronze.crm_cust_info
        FROM '/content/sql-data-warehouse-project/sql-data-warehouse-project/datasets/source_crm/cust_info.csv'
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
        );""")
    cur.execute("""
        COPY bronze.crm_prd_info
        FROM '/content/sql-data-warehouse-project/sql-data-warehouse-project/datasets/source_crm/prd_info.csv'
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
            );""")
    cur.execute("""
        COPY bronze.crm_sales_details
        FROM '/content/sql-data-warehouse-project/sql-data-warehouse-project/datasets/source_crm/sales_details.csv'
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
            );""")
    cur.execute("""
        COPY bronze.erp_cust_az12
        FROM '/content/sql-data-warehouse-project/sql-data-warehouse-project/datasets/source_erp/CUST_AZ12.csv'
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
            );""")
    cur.execute("""
        COPY bronze.erp_loc_a101
        FROM '/content/sql-data-warehouse-project/sql-data-warehouse-project/datasets/source_erp/LOC_A101.csv'
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
            );""")
    cur.execute("""
        COPY bronze.erp_px_cat_g1v2
        FROM '/content/sql-data-warehouse-project/sql-data-warehouse-project/datasets/source_erp/PX_CAT_G1V2.csv'
        WITH (
            FORMAT CSV,
            HEADER TRUE,
            DELIMITER ','
            );""")
conn.commit()


In [ ]:
conn.rollback()
tables = [
    "crm_cust_info",
    "crm_prd_info",
    "crm_sales_details",
    "erp_cust_az12",
    "erp_loc_a101",
    "erp_px_cat_g1v2"
]
with conn.cursor() as cur:
  for table in tables:
    cur.execute(f"""SELECT COUNT(*) FROM bronze.{table};""")
    row_count=cur.fetchone()[0]
    print(f"{table}:{row_count}")

crm_cust_info:18494
crm_prd_info:397
crm_sales_details:60398
erp_cust_az12:18484
erp_loc_a101:18484
erp_px_cat_g1v2:37


In [ ]:
import pandas as pd
df=pd.read_sql_query("SELECT*FROM bronze.crm_cust_info LIMIT 1000",conn)
display(df)

/tmp/ipykernel_1560/4168189889.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("SELECT*FROM bronze.crm_cust_info LIMIT 1000",conn)


,cst_id,cst_key,cst_firstname,cst_lastname,cst_material_status,cst_material_gndr,cst_create_date
0,11000,AW00011000,Jon,Yang,M,M,2025-10-06
1,11001,AW00011001,Eugene,Huang,S,M,2025-10-06
2,11002,AW00011002,Ruben,Torres,M,M,2025-10-06
3,11003,AW00011003,Christy,Zhu,S,F,2025-10-06
4,11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06
...,...,...,...,...,...,...,...
995,11995,AW00011995,Kelvin,Carson,M,None,2025-10-08
996,11996,AW00011996,Veronica,Srini,S,None,2025-10-08
997,11997,AW00011997,Kristina,Kapoor,M,None,2025-10-08
998,11998,AW00011998,Donna,Sharma,M,None,2025-10-08


In [ ]:
#DDL (SILVER LAYER)
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""DROP TABLE IF EXISTS silver.crm_cust_info;""" )
  cur.execute("""CREATE TABLE IF NOT EXISTS silver.crm_cust_info(
                  cst_id INTEGER,
                  cst_key VARCHAR(50),
                  cst_firstname VARCHAR(50),
                  cst_lastname VARCHAR(50),
                  cst_material_status VARCHAR(50),
                  cst_material_gndr VARCHAR(50),
                  cst_create_date DATE,
                  dwh_create_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP);""")

  cur.execute("""DROP TABLE IF EXISTS silver.crm_prd_info;
                 CREATE TABLE IF NOT EXISTS silver.crm_prd_info(
                  prd_id INTEGER,
                  cat_id VARCHAR(50),
                  prd_key VARCHAR(50),
                  prd_nm VARCHAR(50),
                  prd_cost INTEGER,
                  prd_line VARCHAR(50),
                  prd_start_dt DATE,
                  prd_end_dt DATE,
                  dwh_create_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP);""")

  cur.execute("""DROP TABLE IF EXISTS silver.crm_sales_details;
                 CREATE TABLE IF NOT EXISTS silver.crm_sales_details(
                  sls_ord_num VARCHAR(50),
                  sls_prd_key VARCHAR(50),
                  sls_cust_id INTEGER,
                  sls_order_dt DATE,
                  sls_ship_dt DATE,
                  sls_due_dt DATE,
                  sls_sales INTEGER,
                  sls_quantity INTEGER,
                  sls_price INTEGER,
                  dwh_create_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP);""")

  cur.execute("""DROP TABLE IF EXISTS silver.erp_cust_az12;
                 CREATE TABLE IF NOT EXISTS silver.erp_cust_az12(
                  cid VARCHAR(50),
                  bdate DATE,
                  gen VARCHAR(50),
                  dwh_create_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP);""")

  cur.execute("""DROP TABLE IF EXISTS silver.erp_loc_a101;
                 CREATE TABLE IF NOT EXISTS silver.erp_loc_a101(
                  cid VARCHAR(50),
                  cntry VARCHAR(50),
                  dwh_create_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP);""")

  cur.execute("""DROP TABLE IF EXISTS  silver.erp_px_cat_g1v2;
                 CREATE TABLE IF NOT EXISTS silver.erp_px_cat_g1v2(
                  id VARCHAR(50),
                  cat VARCHAR(50),
                  subcat VARCHAR(50),
                  maintenance VARCHAR(50),
                  dwh_create_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP);""")
  conn.commit()

In [ ]:
#Check for nulls or duplicates in primary keys
df=pd.read_sql_query("""SELECT cst_id,COUNT(*)
                 FROM bronze.crm_cust_info
                 GROUP BY cst_id
                 HAVING COUNT(*)>1 OR cst_id IS NULL""",conn)
display(df)

/tmp/ipykernel_1560/3656032645.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT cst_id,COUNT(*)


,cst_id,count
0,NaN,4
1,29473.0,2
2,29449.0,2
3,29433.0,2
4,29466.0,3
5,29483.0,2


In [ ]:
#Cleasnig process
df=pd.read_sql_query("""SELECT*
                        FROM(SELECT * ,ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) as flag_last FROM bronze.crm_cust_info)t
                        WHERE flag_last =1
                        ;""",conn)
display(df)
#Cleansing process2===check for unwanted Spaces
df=pd.read_sql_query("""SELECT prd_nm FROM silver.crm_prd_info WHERE prd_nm != TRIM(prd_nm);""",conn)
display(df)
#Cleansing process2===check for values that is less than 0 or null
df=pd.read_sql_query("""SELECT prd_cost FROM silver.crm_prd_info WHERE prd_cost<0 or prd_cost IS NULL;""",conn)
display(df)
#Check for Invalid Date Orders
df=pd.read_sql_query("""SELECt * FROM silver.crm_prd_info WHERE prd_end_dt < prd_start_dt;""",conn)
display(df)


/tmp/ipykernel_1560/18226892.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT*


,cst_id,cst_key,cst_firstname,cst_lastname,cst_material_status,cst_material_gndr,cst_create_date,flag_last
0,11000.0,AW00011000,Jon,Yang,M,M,2025-10-06,1
1,11001.0,AW00011001,Eugene,Huang,S,M,2025-10-06,1
2,11002.0,AW00011002,Ruben,Torres,M,M,2025-10-06,1
3,11003.0,AW00011003,Christy,Zhu,S,F,2025-10-06,1
4,11004.0,AW00011004,Elizabeth,Johnson,S,F,2025-10-06,1
...,...,...,...,...,...,...,...,...
18480,29480.0,AW00029480,Nina,Raji,S,None,2026-01-25,1
18481,29481.0,AW00029481,Ivan,Suri,S,None,2026-01-25,1
18482,29482.0,AW00029482,Clayton,Zhang,M,None,2026-01-25,1
18483,29483.0,AW00029483,Marc,Navarro,M,None,2026-01-27,1


/tmp/ipykernel_1560/18226892.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT prd_nm FROM silver.crm_prd_info WHERE prd_nm != TRIM(prd_nm);""",conn)


,prd_nm


/tmp/ipykernel_1560/18226892.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT prd_cost FROM silver.crm_prd_info WHERE prd_cost<0 or prd_cost IS NULL;""",conn)


,prd_cost


/tmp/ipykernel_1560/18226892.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECt * FROM silver.crm_prd_info WHERE prd_end_dt < prd_start_dt;""",conn)


,prd_id,cat_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,dwh_create_date


In [ ]:
#Check for unwanted spaces
df=pd.read_sql_query("""SELECT cst_firstname FROM bronze.crm_cust_info WHERE cst_firstname !=trim(cst_firstname);""",conn)
display(df)

/tmp/ipykernel_1560/3206025190.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT cst_firstname FROM bronze.crm_cust_info WHERE cst_firstname !=trim(cst_firstname);""",conn)


,cst_firstname
0,Jon
1,Elizabeth
2,Lauren
3,Ian
4,Chloe
5,Destiny
6,Angela
7,Jessica
8,Caleb
9,Willie


In [ ]:
#Check data standarzation & Consistency
df=pd.read_sql_query("""SELECT DISTINCT prd_line FROM silver.crm_prd_info;""",conn)
display(df)

/tmp/ipykernel_1560/201981727.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT DISTINCT prd_line FROM silver.crm_prd_info;""",conn)


,prd_line


In [ ]:
#DATA Standarzation
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""INSERT INTO silver.crm_cust_info(
                 cst_id,
                 cst_key,
                 cst_firstname,
                 cst_lastname,
                 cst_material_status,
                 cst_material_gndr,
                 cst_create_date)
                 SELECT cst_id,
                        cst_key,
                        TRIM(cst_firstname) AS cst_firstname,
                        TRIM(cst_lastname) AS cst_lastname,
                        CASE WHEN TRIM(UPPER(cst_material_status))= 'S' THEN 'Single'
                             WHEN TRIM(UPPER(cst_material_status))= 'M' THEN 'Married'
                             ELSE 'N/A' END cst_material_status,
                        CASE WHEN TRIM(UPPER(cst_material_gndr))= 'M' THEN 'Male'
                             WHEN TRIM(UPPER(cst_material_gndr))='F' THEN 'Female'
                             ELSE 'N/A' END cst_material_gndr,
                            cst_create_date
                        FROM (SELECT * ,ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) as flag_last FROM bronze.crm_cust_info)t
                        WHERE flag_last =1;""")
conn.commit()
df=pd.read_sql_query("""SELECT cst_id,
                               cst_key,
                               TRIM(cst_firstname) AS cst_firstname,
                               TRIM(cst_lastname) AS cst_lastname,
                               CASE WHEN TRIM(UPPER(cst_material_status))= 'S' THEN 'Single'
                                    WHEN TRIM(UPPER(cst_material_status))= 'M' THEN 'Married'
                                    ELSE 'N/A' END cst_material_status,
                               CASE WHEN TRIM(UPPER(cst_material_gndr))= 'M' THEN 'Male'
                                    WHEN TRIM(UPPER(cst_material_gndr))='F' THEN 'Female'
                                    ELSE 'N/A' END cst_material_gndr,
                               cst_create_date
                        FROM (SELECT * ,ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) as flag_last FROM bronze.crm_cust_info)t
                        WHERE flag_last =1;""",conn)
display(df)

/tmp/ipykernel_1560/3780114641.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT cst_id,


,cst_id,cst_key,cst_firstname,cst_lastname,cst_material_status,cst_material_gndr,cst_create_date
0,11000.0,AW00011000,Jon,Yang,Married,Male,2025-10-06
1,11001.0,AW00011001,Eugene,Huang,Single,Male,2025-10-06
2,11002.0,AW00011002,Ruben,Torres,Married,Male,2025-10-06
3,11003.0,AW00011003,Christy,Zhu,Single,Female,2025-10-06
4,11004.0,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06
...,...,...,...,...,...,...,...
18480,29480.0,AW00029480,Nina,Raji,Single,N/A,2026-01-25
18481,29481.0,AW00029481,Ivan,Suri,Single,N/A,2026-01-25
18482,29482.0,AW00029482,Clayton,Zhang,Married,N/A,2026-01-25
18483,29483.0,AW00029483,Marc,Navarro,Married,N/A,2026-01-27


In [ ]:
#Transforming crm.prd_info
df=pd.read_sql_query("""SELECT prd_id, COUNT(*) FROM bronze.crm_prd_info GROUP BY prd_id HAVING COUNT(*)>1 OR prd_id IS NULL;""",conn)
display(df)
#Not in cat_id
df=pd.read_sql_query("""SELECT prd_id,
                               REPLACE(SUBSTRING(prd_key,1,5),'-','_') as cat_id,
                               REPLACE(SUBSTRING(prd_key,7,length(prd_key)),'-','_') as prd_key,
                               prd_nm,
                               prd_cost,
                               prd_line,
                               prd_start_dt,
                               prd_end_dt
                        FROM  bronze.crm_prd_info
                        WHERE REPLACE(SUBSTRING(prd_key,1,5),'-','_') not in (SELECT DISTINCT id FROM bronze.erp_px_cat_g1v2);""",conn)
display(df)
#Not in prd_key
df=pd.read_sql_query("""SELECT prd_id,
                               REPLACE(SUBSTRING(prd_key,1,5),'-','_') as cat_id,
                               REPLACE(SUBSTRING(prd_key,7,length(prd_key)),'-','_') as prd_key,
                               prd_nm,
                               prd_cost,
                               prd_line,
                               prd_start_dt,
                               prd_end_dt
                        FROM  bronze.crm_prd_info
                        WHERE REPLACE(SUBSTRING(prd_key,7,length(prd_key)),'-','_') not in (SELECT sls_prd_key FROM bronze.crm_sales_details);""",conn)
#Tranforming
df=pd.read_sql_query("""SELECT prd_id,
                               REPLACE(SUBSTRING(prd_key,1,5),'-','_') as cat_id,
                               REPLACE(SUBSTRING(prd_key,7,length(prd_key)),'-','_') as prd_key,
                               prd_nm,
                               COALESCE(prd_cost,0) as prd_cost,
                               CASE WHEN UPPER(TRIM(prd_line)) ='M' THEN 'Mountain'
                                    WHEN UPPER(TRIM(prd_line)) ='R' THEN 'Road'
                                    WHEN UPPER(TRIM(prd_line)) ='S' THEN 'other Sales'
                                    WHEN UPPER(TRIM(prd_line)) ='T' THEN 'Touring'
                                    ELSE 'n/a'
                               END AS prd_line,
                               CAST(prd_start_dt AS DATE) AS prd_start_dt,
                               CAST(LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt)-1 AS DATE) AS  prd_end_dt
                        FROM  bronze.crm_prd_info;""",conn)
display(df)

/tmp/ipykernel_1560/3435332426.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT prd_id, COUNT(*) FROM bronze.crm_prd_info GROUP BY prd_id HAVING COUNT(*)>1 OR prd_id IS NULL;""",conn)


,prd_id,count


/tmp/ipykernel_1560/3435332426.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT prd_id,


,prd_id,cat_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
0,542,CO_PE,PD_M282,LL Mountain Pedal,18,M,2013-07-01,None
1,543,CO_PE,PD_M340,ML Mountain Pedal,28,M,2013-07-01,None
2,544,CO_PE,PD_M562,HL Mountain Pedal,36,M,2013-07-01,None
3,545,CO_PE,PD_R347,LL Road Pedal,18,R,2013-07-01,None
4,546,CO_PE,PD_R563,ML Road Pedal,28,R,2013-07-01,None
5,547,CO_PE,PD_R853,HL Road Pedal,36,R,2013-07-01,None
6,548,CO_PE,PD_T852,Touring Pedal,36,T,2013-07-01,None


/tmp/ipykernel_1560/3435332426.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT prd_id,
/tmp/ipykernel_1560/3435332426.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT prd_id,


,prd_id,cat_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
0,478,AC_BC,BC_M005,Mountain Bottle Cage,4,Mountain,2013-07-01,None
1,479,AC_BC,BC_R205,Road Bottle Cage,3,Road,2013-07-01,None
2,477,AC_BC,WB_H098,Water Bottle - 30 oz.,2,other Sales,2013-07-01,None
3,483,AC_BR,RA_H123,Hitch Rack - 4-Bike,45,other Sales,2013-07-01,None
4,486,AC_BS,ST_1401,All-Purpose Bike Stand,59,Mountain,2013-07-01,None
...,...,...,...,...,...,...,...,...
392,421,CO_WH,RW_M928,HL Mountain Rear Wheel,145,Mountain,2012-07-01,None
393,422,CO_WH,RW_R623,LL Road Rear Wheel,50,Road,2012-07-01,None
394,423,CO_WH,RW_R762,ML Road Rear Wheel,122,Road,2012-07-01,None
395,424,CO_WH,RW_R820,HL Road Rear Wheel,159,Road,2012-07-01,None


In [ ]:
#Checking date and rivsing
df=pd.read_sql_query("""SELECT prd_id,
                        prd_key,
                        prd_nm,
                        prd_start_dt,
                        LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt)-1 as prd_end_dt_test
                        FROM bronze.crm_prd_info
                        WHERE prd_key IN ('AC-HE-HL-U509-R','AC-HE-HL-U509');""",conn)
display(df)


/tmp/ipykernel_1560/1618471825.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT prd_id,


,prd_id,prd_key,prd_nm,prd_start_dt,prd_end_dt_test
0,215,AC-HE-HL-U509,Sport-100 Helmet- Black,2011-07-01,2012-06-30
1,216,AC-HE-HL-U509,Sport-100 Helmet- Black,2012-07-01,2013-06-30
2,217,AC-HE-HL-U509,Sport-100 Helmet- Black,2013-07-01,None
3,212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,2011-07-01,2012-06-30
4,213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,2012-07-01,2013-06-30
5,214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,2013-07-01,None


In [ ]:
#INSERTING THE  QUERY INTO SILVER DDR
with conn.cursor() as cur:
  cur.execute("""INSERT INTO silver.crm_prd_info SELECT prd_id,
                               REPLACE(SUBSTRING(prd_key,1,5),'-','_') as cat_id,
                               REPLACE(SUBSTRING(prd_key,7,length(prd_key)),'-','_') as prd_key,
                               prd_nm,
                               COALESCE(prd_cost,0) as prd_cost,
                               CASE WHEN UPPER(TRIM(prd_line)) ='M' THEN 'Mountain'
                                    WHEN UPPER(TRIM(prd_line)) ='R' THEN 'Road'
                                    WHEN UPPER(TRIM(prd_line)) ='S' THEN 'other Sales'
                                    WHEN UPPER(TRIM(prd_line)) ='T' THEN 'Touring'
                                    ELSE 'n/a'
                               END AS prd_line,
                               CAST(prd_start_dt AS DATE) AS prd_start_dt,
                               CAST(LEAD(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt)-1 AS DATE) AS  prd_end_dt
                        FROM  bronze.crm_prd_info;""",conn)
conn.commit()
df=pd.read_sql_query("""SELECT* FROM silver.crm_prd_info;""",conn)
display(df)

/tmp/ipykernel_1560/63704998.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT* FROM silver.crm_prd_info;""",conn)


,prd_id,cat_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,dwh_create_date
0,478,AC_BC,BC_M005,Mountain Bottle Cage,4,Mountain,2013-07-01,None,2026-11-01 04:17:13.968444
1,479,AC_BC,BC_R205,Road Bottle Cage,3,Road,2013-07-01,None,2026-11-01 04:17:13.968444
2,477,AC_BC,WB_H098,Water Bottle - 30 oz.,2,other Sales,2013-07-01,None,2026-11-01 04:17:13.968444
3,483,AC_BR,RA_H123,Hitch Rack - 4-Bike,45,other Sales,2013-07-01,None,2026-11-01 04:17:13.968444
4,486,AC_BS,ST_1401,All-Purpose Bike Stand,59,Mountain,2013-07-01,None,2026-11-01 04:17:13.968444
...,...,...,...,...,...,...,...,...,...
392,421,CO_WH,RW_M928,HL Mountain Rear Wheel,145,Mountain,2012-07-01,None,2026-11-01 04:17:13.968444
393,422,CO_WH,RW_R623,LL Road Rear Wheel,50,Road,2012-07-01,None,2026-11-01 04:17:13.968444
394,423,CO_WH,RW_R762,ML Road Rear Wheel,122,Road,2012-07-01,None,2026-11-01 04:17:13.968444
395,424,CO_WH,RW_R820,HL Road Rear Wheel,159,Road,2012-07-01,None,2026-11-01 04:17:13.968444


In [ ]:
#silver.crm_sales_details  #WHERE sls_ord_num !=TRIM(sls_ord_num); 를 거침으로써 변화시켜야할 키 있는지 #prd_key -crm_prd_info , cst_id-crm_cust_info
df=pd.read_sql_query("""SELECT sls_ord_num,
                               sls_prd_key,
                               sls_cust_id,
                               CASE WHEN  sls_order_dt=0 or LENGTH(CAST(sls_order_dt AS VARCHAR))!=8 THEN NULL
                                    ELSE CAST(CAST(sls_order_dt AS VARCHAR) AS DATE)
                               END as sls_order_dt,
                               CASE WHEN  sls_ship_dt=0 or LENGTH(CAST(sls_ship_dt AS VARCHAR))!=8 THEN NULL
                                    ELSE CAST(CAST(sls_ship_dt AS VARCHAR) AS DATE)
                               END as sls_ship_dt,
                               CASE WHEN  sls_due_dt=0 or LENGTH(CAST(sls_due_dt AS VARCHAR))!=8 THEN NULL
                                    ELSE CAST(CAST(sls_due_dt AS VARCHAR) AS DATE)
                               END as sls_due_dt,
                               sls_sales,
                               sls_quantity,
                               sls_price
                               FROM bronze.crm_sales_details

                               """,conn)
display(df)

/tmp/ipykernel_1560/2058025172.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT sls_ord_num,


,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
0,SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578.0,1,3578.0
1,SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0
2,SO43699,BK-M82S-44,25863,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0
3,SO43700,BK-R50B-62,14501,2010-12-29,2011-01-05,2011-01-10,699.0,1,699.0
4,SO43701,BK-M82S-44,11003,2010-12-29,2011-01-05,2011-01-10,3400.0,1,3400.0
...,...,...,...,...,...,...,...,...,...
60393,SO75122,FE-6654,15868,2014-01-28,2014-02-04,2014-02-09,22.0,1,22.0
60394,SO75122,CA-1098,15868,2014-01-28,2014-02-04,2014-02-09,9.0,1,9.0
60395,SO75123,FE-6654,18759,2014-01-28,2014-02-04,2014-02-09,22.0,1,22.0
60396,SO75123,ST-1401,18759,2014-01-28,2014-02-04,2014-02-09,159.0,1,159.0


In [ ]:
#---Check for Invalid Dates
conn.rollback()
df=pd.read_sql_query("""SELECT
                        NULLIF(sls_order_dt,0) as sls_order_dt
                        FROM bronze.crm_sales_details
                        WHERE sls_order_dt <=0 or LENGTH(CAST(sls_order_dt AS VARCHAR)) !=8 or sls_order_dt>20500101 OR sls_order_dt<1900101 ; """,conn)
display(df)
df=pd.read_sql_query("""SELECT *
                        FROM silver.crm_sales_details
                        WHERE sls_order_dt> sls_ship_dt OR sls_order_dt > sls_due_dt""",conn)
display(df)

/tmp/ipykernel_1560/450836613.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT


,sls_order_dt
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN
6,NaN
7,NaN
8,NaN
9,NaN


/tmp/ipykernel_1560/450836613.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT *


,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price,dwh_create_date


In [ ]:
#---Check Data Consistency: Between Sales, Quantity, and Price
#>> Sales =Quantity *Price
#>> Values must not be NULL, zero, or negative.
df=pd.read_sql_query("""SELECT
                        sls_sales,
                        sls_quantity,
                        sls_price
                        FROM bronze.crm_sales_details
                        WHERE sls_sales != sls_quantity * sls_price
                        OR sls_sales IS NULL OR sls_quantity IS NULL OR sls_sales <= 0 OR sls_quantity <=0 OR sls_price<=0
                        ORDER BY sls_sales,sls_quantity,sls_price;""",conn)
display(df)
#We Found a lot of mistakes here Like NULL,NEGATIVE, OR ZERO. HERE IS THAT RULES
#1.If sales is negative,zero, or null, derive it using quantity and price
#2. If price is zero, or null, calculate it using sales and quantity
#3. If price is negative, convert it to a positive value
df=pd.read_sql_query("""SELECT
                        sls_sales AS old_sls_sales,
                        sls_quantity AS old_sls_quantity,
                        sls_price AS old_sls_price,
                        CASE WHEN sls_sales IS NULL OR sls_sales <=0 OR sls_sales !=sls_quantity*ABS(sls_price) THEN sls_quantity*ABS(sls_price)
                        ELSE sls_sales END AS sls_sales,
                        CASE WHEN sls_price IS NULL OR sls_price<=0 THEN sls_sales/NULLIF(sls_quantity,0)
                        ELSE sls_price END AS sls_price
                        FROM bronze.crm_sales_details
                        WHERE sls_sales != sls_quantity * ABS(sls_price)
                        OR sls_sales IS NULL OR sls_quantity IS NULL OR sls_sales <= 0 OR sls_quantity <=0 OR sls_price<=0
                        ORDER BY sls_sales,sls_quantity,sls_price;""",conn)
display(df)
df=pd.read_sql_query("""
                 SELECT
                 sls_ord_num,
                 sls_prd_key,
                 sls_cust_id,
                 CASE WHEN sls_order_dt =0 OR LENGTH(CAST(sls_order_dt AS VARCHAR )) !=8 THEN NULL ELSE CAST(CAST(sls_order_dt AS VARCHAR) AS DATE) END AS sls_order_dt,
                 CASE WHEN sls_order_dt =0 OR LENGTH(CAST(sls_ship_dt AS VARCHAR)) !=8 THEN NULL ELSE CAST(CAST(sls_ship_dt AS VARCHAR) AS DATE) END AS sls_ship_dt,
                 CASE WHEN sls_order_dt =0 OR LENGTH(CAST(sls_due_dt AS VARCHAR)) !=8 THEN NULL ELSE CAST(CAST(sls_due_dt AS VARCHAR) AS DATE) END AS sls_due_dt,
                 CASE WHEN sls_sales IS NULL OR sls_sales <=0 OR sls_sales !=sls_quantity*ABS(sls_price) THEN sls_quantity*ABS(sls_price)
                 ELSE sls_sales END AS sls_sales,
                 CASE WHEN sls_price IS NULL OR sls_price<=0 THEN sls_sales/NULLIF(sls_quantity,0)
                 ELSE sls_price END AS sls_price
                 FROM bronze.crm_sales_details
                 WHERE sls_sales != sls_quantity * sls_price
                 OR sls_sales IS NULL OR sls_quantity IS NULL OR sls_sales <= 0 OR sls_quantity <=0 OR sls_price<=0
                 ORDER BY sls_sales,sls_quantity,sls_price;""",conn)


/tmp/ipykernel_1560/256718348.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT


,sls_sales,sls_quantity,sls_price
0,-54.0,1,54
1,-35.0,1,35
2,-18.0,1,9
3,0.0,1,4
4,0.0,1,10
5,2.0,1,50
6,5.0,2,5
7,8.0,1,9
8,21.0,1,-21
9,22.0,1,-22


/tmp/ipykernel_1560/256718348.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT


,old_sls_sales,old_sls_quantity,old_sls_price,sls_sales,sls_price
0,40.0,1,2,2,2
1,NaN,1,2,2,2
2,0.0,1,4,4,4
3,NaN,1,8,8,8
4,8.0,1,9,9,9
5,NaN,1,9,9,9
6,-18.0,1,9,9,9
7,0.0,1,10,10,10
8,NaN,1,10,10,10
9,5.0,2,5,10,5


/tmp/ipykernel_1560/256718348.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""


In [ ]:
#INSERTING
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""INSERT INTO silver.crm_sales_details
                 SELECT
                 sls_ord_num,
                 sls_prd_key,
                 sls_cust_id,
                 CASE WHEN sls_order_dt =0 OR LENGTH(CAST(sls_order_dt AS VARCHAR )) !=8 THEN NULL ELSE CAST(CAST(sls_order_dt AS VARCHAR) AS DATE) END AS sls_order_dt,
                 CASE WHEN sls_order_dt =0 OR LENGTH(CAST(sls_ship_dt AS VARCHAR)) !=8 THEN NULL ELSE CAST(CAST(sls_ship_dt AS VARCHAR) AS DATE) END AS sls_ship_dt,
                 CASE WHEN sls_order_dt =0 OR LENGTH(CAST(sls_due_dt AS VARCHAR)) !=8 THEN NULL ELSE CAST(CAST(sls_due_dt AS VARCHAR) AS DATE) END AS sls_due_dt,
                 CASE WHEN sls_sales IS NULL OR sls_sales <=0 OR sls_sales !=sls_quantity*ABS(sls_price) THEN sls_quantity*ABS(sls_price)
                 ELSE sls_sales END AS sls_sales,
                 CASE WHEN sls_price IS NULL OR sls_price<=0 THEN sls_sales/NULLIF(sls_quantity,0)
                 ELSE sls_price END AS sls_price
                 FROM bronze.crm_sales_details
                 WHERE sls_sales != sls_quantity * sls_price
                 OR sls_sales IS NULL OR sls_quantity IS NULL OR sls_sales <= 0 OR sls_quantity <=0 OR sls_price<=0
                 ORDER BY sls_sales,sls_quantity,sls_price;""")
  conn.commit()
  #SEE
  df=pd.read_sql_query("""SELECT* FROM silver.crm_sales_details;""",conn)
  display(df)

/tmp/ipykernel_1560/4135373911.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT* FROM silver.crm_sales_details;""",conn)


,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price,dwh_create_date
0,SO67993,PK-7098,17084,2013-10-08,2013-10-15,2013-10-20,2,2,None,2026-11-01 04:17:18.181427
1,SO68588,PK-7098,11318,2013-10-17,2013-10-24,2013-10-29,2,2,None,2026-11-01 04:17:18.181427
2,SO68889,TT-R982,14086,2013-10-22,2013-10-29,2013-11-03,4,4,None,2026-11-01 04:17:18.181427
3,SO61561,CL-9009,18380,2013-07-05,2013-07-12,2013-07-17,8,8,None,2026-11-01 04:17:18.181427
4,SO68551,BC-R205,22691,2013-10-17,2013-10-24,2013-10-29,9,9,None,2026-11-01 04:17:18.181427
5,SO61548,CA-1098,12386,2013-07-05,2013-07-12,2013-07-17,9,9,None,2026-11-01 04:17:18.181427
6,SO61570,CA-1098,17809,2013-07-05,2013-07-12,2013-07-17,9,9,None,2026-11-01 04:17:18.181427
7,SO61636,BC-M005,12914,2013-07-06,2013-07-13,2013-07-18,10,10,None,2026-11-01 04:17:18.181427
8,SO67551,BC-M005,15150,2013-10-01,2013-10-08,2013-10-13,10,10,None,2026-11-01 04:17:18.181427
9,SO67707,TT-M928,26046,2013-10-04,2013-10-11,2013-10-16,10,5,None,2026-11-01 04:17:18.181427


In [ ]:
#TABLE erp_cust_za12
df=pd.read_sql_query("""SELECT cid,bdate,gen FROM bronze.erp_cust_az12 WHERE cid LIKE '%AW00011000%'""",conn)
display(df)
df=pd.read_sql_query("""SELECT* FROM silver.crm_cust_info;""",conn)
display(df)
#WE HAVE UNNECESSARY CHARACTERS 'NAS' RIGHT INFRONT OF CID COLUMN. THEREFORE WE NEED TO SPLIT IT
#AFTER THE TRANSFORMATION, THERE'S NO UNMATCHING DATA
df=pd.read_sql_query("""SELECT cid,
                        CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid,4,LENGTH(cid))
                        ELSE cid END AS cid,
                        bdate,
                        gen
                        FROM bronze.erp_cust_az12
                        WHERE  CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid,4,LENGTH(cid)) ELSE cid END NOT IN (SELECT DISTINCT cst_key FROM silver.crm_cust_info);""",conn )
display(df)
#Transforming
df=pd.read_sql_query("""SELECT
                        CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid,4,LENGTH(cid))
                        ELSE cid END AS cid,
                        CASE WHEN bdate > CURRENT_DATE THEN NULL ELSE bdate END AS bdate,
                        CASE WHEN UPPER(TRIM(gen)) IN ('F','FEMALE') THEN 'Female'
                             WHEN UPPER(TRIM(gen)) IN ('M','MALE') THEN 'Male'
                             ELSE 'N/A' END AS gen
                        FROM bronze.erp_cust_az12;""",conn)

display(df)
#---Identify Out-of-Range Dates
df=pd.read_sql_query("""SELECT bdate
                        FROM silver.erp_cust_az12
                        WHERE bdate<'1924-01-01'OR bdate >CURRENT_DATE ;""",conn)#100years old limit and check for birthdays in the future
display(df)
#--Data Standardization & Consistency we found 5 values -->3 values (male, female,null)
df=pd.read_sql_query("""SELECT DISTINCT gen FROM silver.erp_cust_az12;""",conn)
display(df)
#Inserting -REMOVE 'NAS' prefix if present, set future birthdates to null, normalize gender values and handle unknown cases
with conn.cursor() as cur:
  cur.execute("""INSERT INTO silver.erp_cust_az12
                        SELECT
                        CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid,4,LENGTH(cid))
                        ELSE cid END AS cid,
                        CASE WHEN bdate > CURRENT_DATE THEN NULL ELSE bdate END AS bdate,
                        CASE WHEN UPPER(TRIM(gen)) IN ('F','FEMALE') THEN 'Female'
                             WHEN UPPER(TRIM(gen)) IN ('M','MALE') THEN 'Male'
                             ELSE 'N/A' END AS gen
                        FROM bronze.erp_cust_az12;""")
conn.commit()
#RESULT
df=pd.read_sql_query("""SELECT* FROM silver.erp_cust_az12;""",conn)
display(df)

/tmp/ipykernel_1560/4169631146.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT cid,bdate,gen FROM bronze.erp_cust_az12 WHERE cid LIKE '%AW00011000%'""",conn)


,cid,bdate,gen
0,NASAW00011000,1971-10-06,Male


/tmp/ipykernel_1560/4169631146.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT* FROM silver.crm_cust_info;""",conn)


,cst_id,cst_key,cst_firstname,cst_lastname,cst_material_status,cst_material_gndr,cst_create_date,dwh_create_date
0,11000.0,AW00011000,Jon,Yang,Married,Male,2025-10-06,2026-11-01 04:17:13.678216
1,11001.0,AW00011001,Eugene,Huang,Single,Male,2025-10-06,2026-11-01 04:17:13.678216
2,11002.0,AW00011002,Ruben,Torres,Married,Male,2025-10-06,2026-11-01 04:17:13.678216
3,11003.0,AW00011003,Christy,Zhu,Single,Female,2025-10-06,2026-11-01 04:17:13.678216
4,11004.0,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06,2026-11-01 04:17:13.678216
...,...,...,...,...,...,...,...,...
18480,29480.0,AW00029480,Nina,Raji,Single,N/A,2026-01-25,2026-11-01 04:17:13.678216
18481,29481.0,AW00029481,Ivan,Suri,Single,N/A,2026-01-25,2026-11-01 04:17:13.678216
18482,29482.0,AW00029482,Clayton,Zhang,Married,N/A,2026-01-25,2026-11-01 04:17:13.678216
18483,29483.0,AW00029483,Marc,Navarro,Married,N/A,2026-01-27,2026-11-01 04:17:13.678216


/tmp/ipykernel_1560/4169631146.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT cid,


,cid,cid,bdate,gen


... (4 outputs left out)

/tmp/ipykernel_1560/4169631146.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT DISTINCT gen FROM silver.erp_cust_az12;""",conn)


,gen


/tmp/ipykernel_1560/4169631146.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT* FROM silver.erp_cust_az12;""",conn)


,cid,bdate,gen,dwh_create_date
0,AW00011000,1971-10-06,Male,2026-11-01 04:17:18.249266
1,AW00011001,1976-05-10,Male,2026-11-01 04:17:18.249266
2,AW00011002,1971-02-09,Male,2026-11-01 04:17:18.249266
3,AW00011003,1973-08-14,Female,2026-11-01 04:17:18.249266
4,AW00011004,1979-08-05,Female,2026-11-01 04:17:18.249266
...,...,...,...,...
18479,AW00029479,1969-06-30,N/A,2026-11-01 04:17:18.249266
18480,AW00029480,1977-05-06,N/A,2026-11-01 04:17:18.249266
18481,AW00029481,1965-07-04,N/A,2026-11-01 04:17:18.249266
18482,AW00029482,1964-09-01,N/A,2026-11-01 04:17:18.249266


In [ ]:
#Table erp_loc_a101 we have to match cid in the table with crm_cust_inso's cst_key--Normalize and handle missing or blank country codes
df=pd.read_sql_query("""SELECT
                        REPLACE(cid,'-','') AS cid,
                        CASE WHEN TRIM(cntry) ='DE' THEN 'Germany'
                             WHEN TRIM(cntry) IN ('US','USA') THEN 'United States'
                             WHEN TRIM(cntry) ='' or cntry IS NULL THEN 'N/A'
                             ELSE TRIM(cntry)
                        END AS cntry
                        FROM bronze.erp_loc_a101""",conn)
display(df)
df=pd.read_sql_query("""SELECT cst_key FROM silver.crm_cust_info;""",conn)
display(df)
#DATA STANDARDIZATION & CONSISTENCY
df=pd.read_sql_query("""SELECT DISTINCT cntry FROM bronze.erp_loc_a101 ORDER BY cntry;""",conn)
display(df)
#INSERT
with conn.cursor() as cur:
  cur.execute("""INSERT INTO silver.erp_loc_a101
                        SELECT
                        REPLACE(cid,'-','') AS cid,
                        CASE WHEN TRIM(cntry) ='DE' THEN 'Germany'
                             WHEN TRIM(cntry) IN ('US','USA') THEN 'United States'
                             WHEN TRIM(cntry) ='' or cntry IS NULL THEN 'N/A'
                             ELSE TRIM(cntry)
                        END AS cntry
                        FROM bronze.erp_loc_a101;""")
conn.commit()

/tmp/ipykernel_1560/1668596814.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT


,cid,cntry
0,AW00011000,Australia
1,AW00011001,Australia
2,AW00011002,Australia
3,AW00011003,Australia
4,AW00011004,Australia
...,...,...
18479,AW00029479,France
18480,AW00029480,United Kingdom
18481,AW00029481,Germany
18482,AW00029482,France


/tmp/ipykernel_1560/1668596814.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT cst_key FROM silver.crm_cust_info;""",conn)


,cst_key
0,AW00011000
1,AW00011001
2,AW00011002
3,AW00011003
4,AW00011004
...,...
18480,AW00029480
18481,AW00029481
18482,AW00029482
18483,AW00029483


/tmp/ipykernel_1560/1668596814.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT DISTINCT cntry FROM bronze.erp_loc_a101 ORDER BY cntry;""",conn)


,cntry
0,
1,
2,
3,Australia
4,Canada
5,DE
6,France
7,Germany
8,United Kingdom
9,United States


In [ ]:
conn.rollback()

In [ ]:
#Table erp_px_cat_g1v2
df=pd.read_sql_query("""SELECT id,cat,subcat,maintenance FROM bronze.erp_px_cat_g1v2;""",conn)
display(df)
#WE NEED TO MATCH ITS FIRST COL WITH prd_key from crm_prd_info (matched(already))
#--Check for unwanted Spaces
df=pd.read_sql_query("""SELECT* FROM bronze.erp_px_cat_g1v2 WHERE cat !=TRIM(cat) OR subcat !=TRIM(subcat) OR maintenance !=TRIM(maintenance);""",conn)
display(df)
#---Data Standardization & Consistency
df=pd.read_sql_query("""SELECT DISTINCT maintenance FROM bronze.erp_px_cat_g1v2;""",conn)
display(df)
#INSERT

with conn.cursor() as cur:
  cur.execute("""INSERT INTO silver.erp_px_cat_g1v2
                 SELECT id,
                        cat,
                        subcat,
                        maintenance
                 FROM bronze.erp_px_cat_g1v2;"""
                 )
conn.commit()


/tmp/ipykernel_1560/3434469534.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT id,cat,subcat,maintenance FROM bronze.erp_px_cat_g1v2;""",conn)


,id,cat,subcat,maintenance
0,AC_BR,Accessories,Bike Racks,Yes
1,AC_BS,Accessories,Bike Stands,No
2,AC_BC,Accessories,Bottles and Cages,No
3,AC_CL,Accessories,Cleaners,Yes
4,AC_FE,Accessories,Fenders,No
5,AC_HE,Accessories,Helmets,Yes
6,AC_HP,Accessories,Hydration Packs,No
7,AC_LI,Accessories,Lights,Yes
8,AC_LO,Accessories,Locks,Yes
9,AC_PA,Accessories,Panniers,No


/tmp/ipykernel_1560/3434469534.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT* FROM bronze.erp_px_cat_g1v2 WHERE cat !=TRIM(cat) OR subcat !=TRIM(subcat) OR maintenance !=TRIM(maintenance);""",conn)


,id,cat,subcat,maintenance


/tmp/ipykernel_1560/3434469534.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT DISTINCT maintenance FROM bronze.erp_px_cat_g1v2;""",conn)


,maintenance
0,No
1,Yes


Insert 전에
TRUNCATE: 기존 테이블 데이터를 전부 삭제한다.
LOAD: 원본(Bronze)에서 데이터를 다시 가져와 채운다.
목적: 데이터 중복을 방지하고 최신 데이터로 깔끔하게 다시 적재하기 위해서다.

In [ ]:
#TRUNCATE AND MAKE IR AS A PROCEDURE\#Similar to Bronze, enhance the output of Silver stored procedure
with conn.cursor() as cur:
    cur.execute("""
    CREATE OR REPLACE PROCEDURE silver.load_silver()
    LANGUAGE plpgsql
    AS $$
    DECLARE
        start_time TIMESTAMP;
        end_time TIMESTAMP;
        batch_start_time TIMESTAMP;
        batch_end_time TIMESTAMP;
    BEGIN
        batch_start_time := clock_timestamp();
        RAISE NOTICE '=======================================';
        RAISE NOTICE 'Loading Silver Layer';
        RAISE NOTICE '=======================================';

        RAISE NOTICE '=======================================';
        RAISE NOTICE 'Loading CRM Tables';
        RAISE NOTICE '=======================================';

        start_time := clock_timestamp();
        RAISE NOTICE 'Truncating Table: silver.crm_cust_info';
        TRUNCATE TABLE silver.crm_cust_info;

        RAISE NOTICE 'Inserting Table: silver.crm_cust_info';
        INSERT INTO silver.crm_cust_info(
            cst_id,
            cst_key,
            cst_firstname,
            cst_lastname,
            cst_material_status,
            cst_material_gndr,
            cst_create_date
        )
        SELECT cst_id,
               cst_key,
               TRIM(cst_firstname) AS cst_firstname,
               TRIM(cst_lastname) AS cst_lastname,
               CASE WHEN TRIM(UPPER(cst_material_status)) = 'S' THEN 'Single'
                    WHEN TRIM(UPPER(cst_material_status)) = 'M' THEN 'Married'
                    ELSE 'N/A' END cst_material_status,
               CASE WHEN TRIM(UPPER(cst_material_gndr)) = 'M' THEN 'Male'
                    WHEN TRIM(UPPER(cst_material_gndr)) = 'F' THEN 'Female'
                    ELSE 'N/A' END cst_material_gndr,
               cst_create_date
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY cst_id
                       ORDER BY cst_create_date DESC
                   ) AS flag_last
            FROM bronze.crm_cust_info
        ) t
        WHERE flag_last = 1;

        end_time := clock_timestamp();
        RAISE NOTICE '>>Load Duration: % seconds',
            EXTRACT(EPOCH FROM (end_time - start_time))::INTEGER;

        start_time := clock_timestamp();
        RAISE NOTICE 'Truncating Table: silver.crm_prd_info';
        TRUNCATE TABLE silver.crm_prd_info;

        RAISE NOTICE 'Inserting Table: silver.crm_prd_info';
        INSERT INTO silver.crm_prd_info
        SELECT prd_id,
               REPLACE(SUBSTRING(prd_key,1,5),'-','_') AS cat_id,
               REPLACE(SUBSTRING(prd_key,7,LENGTH(prd_key)),'-','_') AS prd_key,
               prd_nm,
               COALESCE(prd_cost,0) AS prd_cost,
               CASE WHEN UPPER(TRIM(prd_line)) = 'M' THEN 'Mountain'
                    WHEN UPPER(TRIM(prd_line)) = 'R' THEN 'Road'
                    WHEN UPPER(TRIM(prd_line)) = 'S' THEN 'other Sales'
                    WHEN UPPER(TRIM(prd_line)) = 'T' THEN 'Touring'
                    ELSE 'n/a'
               END AS prd_line,
               CAST(prd_start_dt AS DATE) AS prd_start_dt,
               CAST(
                   LEAD(prd_start_dt) OVER (
                       PARTITION BY prd_key
                       ORDER BY prd_start_dt
                   ) - 1 AS DATE
               ) AS prd_end_dt
        FROM bronze.crm_prd_info;

        end_time := clock_timestamp();
        RAISE NOTICE '>> Load Duration: % seconds',
            EXTRACT(EPOCH FROM (end_time - start_time))::INTEGER;

        start_time := clock_timestamp();
        RAISE NOTICE 'Truncating Table: silver.crm_sales_details';
        TRUNCATE TABLE silver.crm_sales_details;

        RAISE NOTICE 'Inserting Table: silver.crm_sales_details';
        INSERT INTO silver.crm_sales_details
        SELECT
            sls_ord_num,
            sls_prd_key,
            sls_cust_id,
            CASE WHEN sls_order_dt = 0 OR LENGTH(CAST(sls_order_dt AS VARCHAR)) != 8
                 THEN NULL
                 ELSE CAST(CAST(sls_order_dt AS VARCHAR) AS DATE)
            END AS sls_order_dt,
            CASE WHEN sls_order_dt = 0 OR LENGTH(CAST(sls_ship_dt AS VARCHAR)) != 8
                 THEN NULL
                 ELSE CAST(CAST(sls_ship_dt AS VARCHAR) AS DATE)
            END AS sls_ship_dt,
            CASE WHEN sls_order_dt = 0 OR LENGTH(CAST(sls_due_dt AS VARCHAR)) != 8
                 THEN NULL
                 ELSE CAST(CAST(sls_due_dt AS VARCHAR) AS DATE)
            END AS sls_due_dt,
            CASE WHEN sls_sales IS NULL OR sls_sales <= 0
                      OR sls_sales != sls_quantity * ABS(sls_price)
                 THEN sls_quantity * ABS(sls_price)
                 ELSE sls_sales
            END AS sls_sales,
            CASE WHEN sls_price IS NULL OR sls_price <= 0
                 THEN sls_sales / NULLIF(sls_quantity,0)
                 ELSE sls_price
            END AS sls_price
        FROM bronze.crm_sales_details
        WHERE sls_sales != sls_quantity * sls_price
           OR sls_sales IS NULL
           OR sls_quantity IS NULL
           OR sls_sales <= 0
           OR sls_quantity <= 0
           OR sls_price <= 0
        ORDER BY sls_sales, sls_quantity, sls_price;

        end_time := clock_timestamp();
        RAISE NOTICE '>> Load Duration: % seconds',
            EXTRACT(EPOCH FROM (end_time - start_time))::INTEGER;

        start_time := clock_timestamp();
        RAISE NOTICE 'Truncating Table: silver.erp_cust_az12';
        TRUNCATE TABLE silver.erp_cust_az12;

        RAISE NOTICE 'Inserting Table: silver.erp_cust_az12';
        INSERT INTO silver.erp_cust_az12
        SELECT
            CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid,4,LENGTH(cid))
                 ELSE cid
            END AS cid,
            CASE WHEN bdate > CURRENT_DATE THEN NULL
                 ELSE bdate
            END AS bdate,
            CASE WHEN UPPER(TRIM(gen)) IN ('F','FEMALE') THEN 'Female'
                 WHEN UPPER(TRIM(gen)) IN ('M','MALE') THEN 'Male'
                 ELSE 'N/A'
            END AS gen
        FROM bronze.erp_cust_az12;

        end_time := clock_timestamp();
        RAISE NOTICE '>> Load Duration: % seconds',
            EXTRACT(EPOCH FROM (end_time - start_time))::INTEGER;

        start_time := clock_timestamp();
        RAISE NOTICE 'Truncating Table: silver.erp_loc_a101';
        TRUNCATE TABLE silver.erp_loc_a101;

        RAISE NOTICE 'Inserting Table: silver.erp_loc_a101';
        INSERT INTO silver.erp_loc_a101
        SELECT
            REPLACE(cid,'-','') AS cid,
            CASE WHEN TRIM(cntry) = 'DE' THEN 'Germany'
                 WHEN TRIM(cntry) IN ('US','USA') THEN 'United States'
                 WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'N/A'
                 ELSE TRIM(cntry)
            END AS cntry
        FROM bronze.erp_loc_a101;

        end_time := clock_timestamp();
        RAISE NOTICE '>> Load Duration: % seconds',
            EXTRACT(EPOCH FROM (end_time - start_time))::INTEGER;

        start_time := clock_timestamp();
        RAISE NOTICE 'Truncating Table: silver.erp_px_cat_g1v2';
        TRUNCATE TABLE silver.erp_px_cat_g1v2;

        RAISE NOTICE 'Inserting Table: silver.erp_px_cat_g1v2';
        INSERT INTO silver.erp_px_cat_g1v2
        SELECT id,
               cat,
               subcat,
               maintenance
        FROM bronze.erp_px_cat_g1v2;

        end_time := clock_timestamp();
        RAISE NOTICE '>> Load Duration: % seconds',
            EXTRACT(EPOCH FROM (end_time - start_time))::INTEGER;

    END;
    $$;
    """)
conn.commit()

In [ ]:
conn.rollback()
#conn.notices.clear()
#with conn.cursor() as cur:
 # cur.execute("CALL bronze.truncate_bronze();")
  #conn.commit()
#print("".join(conn.notices))
with conn.cursor() as cur:
  cur.execute("CALL silver.load_silver();")
  conn.commit()
print("".join(conn.notices))

NOTICE:  CRM tables truncated
NOTICE:  CRM truncate duration: 0.001 seconds
NOTICE:  ERP tables truncated
NOTICE:  ERP truncate duration: 0.001 seconds
NOTICE:  CRM tables truncated
NOTICE:  CRM truncate duration: 0.001 seconds
NOTICE:  ERP tables truncated
NOTICE:  ERP truncate duration: 0.000 seconds
NOTICE:  table "crm_cust_info" does not exist, skipping
NOTICE:  table "crm_prd_info" does not exist, skipping
NOTICE:  table "crm_sales_details" does not exist, skipping
NOTICE:  table "erp_cust_az12" does not exist, skipping
NOTICE:  table "erp_loc_a101" does not exist, skipping
NOTICE:  table "erp_px_cat_g1v2" does not exist, skipping
NOTICE:  =======================================
NOTICE:  Loading Silver Layer
NOTICE:  =======================================
NOTICE:  =======================================
NOTICE:  Loading CRM Tables
NOTICE:  =======================================
NOTICE:  Truncating Table: silver.crm_cust_info
NOTICE:  Inserting Table: silver.crm_cust_info
NOTICE:

#DATA
#Transformation
:Data Enrichment, Data Integration, Derived Columns, Data Aggregations, Bisiness Rules&Logic, Data Normalization & Standardization,#Data Cleansing
#Data cleansing
Remove Duplicates, Data Filtering, Handling Missing Data, Handling Invalid Values, Handling unwanted spaces, Data Type Casting, Outlier Detection

Gold Layer
1.Analyising :Explore & Understand the Business Objects
2.Coding: Data Integration (Build the Business Object>>Choose Type Dimension vs Fact>>Rename to friendlt names)
3.Validating:Data Integration Checks
4.Docs & Version:Documenting Versioning in GIT

**DATA MODELING**
From raw data ->(organizing and structering in meaningful ways)->putting data into objects that each one of them focus on some specific info. we have to describe their relationships among them.
**Three different processes**
1.Conceptual Data model: entity(customers, oders, and products tables) what are the entities that we have and relationship between them(BIG PICTURE)
2.Logical Data Model:Specifying what are the different columns that we can find in each entity. Drawing the relationship between those, and making it clear which columns are the primary key.(not worring how exactly we need to store those tables in the batabase)(BLUE PRINT)
3.Pysical Data Model: everything gets ready before creating it in the database. we are going to add all the technical details like adding for each column the data types and length of each data type.
(IMPLEMENTATION)

**Star Schema **
Fact talbe containg events or transactions in the middle and it's surrounded by dimensions which contain descriptive info.
Simple, and easy, but we might have duplicates data in more than two dimentions
**SnowFlake Schema**
Similar to star schema. In the middle, we have a fact table and dimensions surrounding the main fact table. in addition to that we have subdimensions surrounding each demension. Thus, all items make it look like a snow flake.
Complex, but notmalization as u break those redundanceis in small tables u can optimize the storage

#STAR SCHEMA
DIMENSION(product details)
Descriptive information that give context to your data.
NOMALY TABLES THAT ANSWER WHO,WHAT,WHERE THOSE ARE DIMENSION
FACT(TRANSACTION)
Quantitative information that represent events.
like if you have multiples like dates, multiple products, and numbers then you can think that is the fact table.
HOW MUCH??, HOW MANY?? FACE TALBE CONTAIN LIEK THEM


# Gold Layer
**Definition** :Business-Ready data
**Objectove**:Provide data to be consumed for reporting & Analytics
Object Type: Views
**Load Method**: None
**Data Transformation**: Data Integration, Data Aggregation, and Business Logic & Rules
**Data Modeling**: Start Schema, Aggregated Objects, and Flat Tables
Target Audience: Data Analysts, Business Users

In [ ]:
conn.rollback()
df=pd.read_sql_query("""SELECT* FROM silver.crm_cust_info""",conn)
display(df)

/tmp/ipykernel_1560/2251592711.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql_query("""SELECT* FROM silver.crm_cust_info""",conn)


,cst_id,cst_key,cst_firstname,cst_lastname,cst_material_status,cst_material_gndr,cst_create_date,dwh_create_date
0,11000.0,AW00011000,Jon,Yang,Married,Male,2025-10-06,2026-11-01 04:17:21.562967
1,11001.0,AW00011001,Eugene,Huang,Single,Male,2025-10-06,2026-11-01 04:17:21.562967
2,11002.0,AW00011002,Ruben,Torres,Married,Male,2025-10-06,2026-11-01 04:17:21.562967
3,11003.0,AW00011003,Christy,Zhu,Single,Female,2025-10-06,2026-11-01 04:17:21.562967
4,11004.0,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06,2026-11-01 04:17:21.562967
...,...,...,...,...,...,...,...,...
18480,29480.0,AW00029480,Nina,Raji,Single,N/A,2026-01-25,2026-11-01 04:17:21.562967
18481,29481.0,AW00029481,Ivan,Suri,Single,N/A,2026-01-25,2026-11-01 04:17:21.562967
18482,29482.0,AW00029482,Clayton,Zhang,Married,N/A,2026-01-25,2026-11-01 04:17:21.562967
18483,29483.0,AW00029483,Marc,Navarro,Married,N/A,2026-01-27,2026-11-01 04:17:21.562967


In [ ]:
#From Silver to Gold
#After Joining table, check if any duplicates were introduced by the join logic using 'group by'
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""SELECT  ci.cst_id,
                         ci.cst_key ,
                         ci.cst_firstname ,
                         ci.cst_lastname,
                         ci.cst_material_status,
                         ci.cst_material_gndr,
                         ci.cst_create_date,
                         ca.bdate,
                         ca.gen,
                         la.cntry as cntry
                  FROM silver.crm_cust_info ci
                  LEFT JOIN silver.erp_cust_az12 ca ON ci.cst_key=ca.cid
                  LEFT JOIN silver.erp_loc_a101 la ON ci.cst_key = la.cid;""")
  columns = [desc[0] for desc in cur.description]
  rows = cur.fetchall()

  import pandas as pd
  df = pd.DataFrame(rows, columns=columns)
  display(df)

  cur.execute(""" SELECT cst_id, COUNT(*) FROM(SELECT ci.cst_id,
                         ci.cst_key,
                         ci.cst_firstname,
                         ci.cst_lastname,
                         ci.cst_material_status,
                         ci.cst_material_gndr,
                         ci.cst_create_date,
                         ca.bdate,
                         ca.gen,
                         la.cntry
                  FROM silver.crm_cust_info ci
                  LEFT JOIN silver.erp_cust_az12 ca ON ci.cst_key=ca.cid
                  LEFT JOIN silver.erp_loc_a101 la ON ci.cst_key = la.cid)t
                  GROUP BY cst_id
                  HAVING COUNT(*)>1;""")
  conn.commit()
  display(pd.DataFrame(cur.fetchall()))

,cst_id,cst_key,cst_firstname,cst_lastname,cst_material_status,cst_material_gndr,cst_create_date,bdate,gen,cntry
0,NaN,A01Ass,None,None,N/A,N/A,None,None,None,None
1,11000.0,AW00011000,Jon,Yang,Married,Male,2025-10-06,1971-10-06,Male,Australia
2,11001.0,AW00011001,Eugene,Huang,Single,Male,2025-10-06,1976-05-10,Male,Australia
3,11002.0,AW00011002,Ruben,Torres,Married,Male,2025-10-06,1971-02-09,Male,Australia
4,11003.0,AW00011003,Christy,Zhu,Single,Female,2025-10-06,1973-08-14,Female,Australia
...,...,...,...,...,...,...,...,...,...,...
18480,29479.0,AW00029479,Tommy,Tang,Married,N/A,2026-01-25,1969-06-30,N/A,France
18481,29480.0,AW00029480,Nina,Raji,Single,N/A,2026-01-25,1977-05-06,N/A,United Kingdom
18482,29481.0,AW00029481,Ivan,Suri,Single,N/A,2026-01-25,1965-07-04,N/A,Germany
18483,29482.0,AW00029482,Clayton,Zhang,Married,N/A,2026-01-25,1964-09-01,N/A,France


""


In [ ]:
#The problem is that we have two gender columns so we have to make a wise decision
#NOTE: null(none) often come from joined tables! Null will appear if SQL finds no match
#QUESTION: FEMALE VS MALE WHICH SOURCE IS THE MASTER FOR THESE VALUE:THE MASTER SOURCE OF CUSTOMER DATE IS CRM
#WE NEED TO DO DATA INTEGRATION
conn.rollback()
with conn.cursor() as cur:
    cur.execute("""
        SELECT DISTINCT
            ci.cst_material_gndr AS cst_gndr,
            ca.gen AS gen,
            CASE WHEN  ci.cst_material_gndr !='N/A' THEN  ci.cst_material_gndr
            ELSE COALESCE(ca.gen,'N/A') END AS new_gen
        FROM silver.crm_cust_info ci
        LEFT JOIN silver.erp_cust_az12 ca
            ON ci.cst_key = ca.cid
        LEFT JOIN silver.erp_loc_a101 la
            ON ci.cst_key = la.cid
        ORDER BY 1, 2;
    """)

    columns = [desc[0] for desc in cur.description]
    rows = cur.fetchall()

df = pd.DataFrame(rows, columns=columns)
display(df)

conn.commit()

,cst_gndr,gen,new_gen
0,Female,Female,Female
1,Female,Male,Female
2,Female,N/A,Female
3,Male,Female,Male
4,Male,Male,Male
5,Male,N/A,Male
6,N/A,Female,Female
7,N/A,Male,Male
8,N/A,N/A,N/A
9,N/A,None,N/A


In [ ]:
#nameing, sorting the columns into logical groups to improve readability
#Since this table doesn't measure anything nor show transactions, therefore this is dimension
#If we want to make a dimension, we always need a primary key for the dimension
#Primary key:we call it 'Surrogate Key' which is system-generated unique identifier assigned to each record in a table, making each record unique
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""SELECT
                         ROW_NUMBER() OVER (ORDER BY cst_id) AS customer_key,
                         ci.cst_id AS customer_id,
                         ci.cst_key AS customer_number,
                         ci.cst_firstname AS first_name,
                         ci.cst_lastname AS last_name,
                         la.cntry AS country,
                         ci.cst_material_status AS marital_status,
                         CASE WHEN  ci.cst_material_gndr !='N/A' THEN  ci.cst_material_gndr
            ELSE COALESCE(ca.gen,'N/A') END AS gender,
                         ca.bdate AS birthdate,
                         ci.cst_create_date
                  FROM silver.crm_cust_info ci
                  LEFT JOIN silver.erp_cust_az12 ca ON ci.cst_key=ca.cid
                  LEFT JOIN silver.erp_loc_a101 la ON ci.cst_key = la.cid;""")
  columns = [desc[0] for desc in cur.description]
  rows = cur.fetchall()

  import pandas as pd
  df = pd.DataFrame(rows, columns=columns)
  display(df)

,customer_key,customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,cst_create_date
0,1,11000.0,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
1,2,11001.0,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
2,3,11002.0,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
3,4,11003.0,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
4,5,11004.0,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06
...,...,...,...,...,...,...,...,...,...,...
18480,18481,29480.0,AW00029480,Nina,Raji,United Kingdom,Single,N/A,1977-05-06,2026-01-25
18481,18482,29481.0,AW00029481,Ivan,Suri,Germany,Single,N/A,1965-07-04,2026-01-25
18482,18483,29482.0,AW00029482,Clayton,Zhang,France,Married,N/A,1964-09-01,2026-01-25
18483,18484,29483.0,AW00029483,Marc,Navarro,France,Married,N/A,1965-06-06,2026-01-27


# General Principles
1. Naming Conventions: Use snake_case, with lowercase letters and underscores(_) to seperate words.
2.Language: Use English for all names.
3.Avoid Reserved Words: Do not use SQL reserved words as object names.


ALL THE OBJECTS WE DICIDED TO MAKE IN THE GOLD LAYER ARE GOING TO BE VIRTUAL ONES

In [ ]:
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""
                  CREATE OR REPLACE VIEW gold.dim_customers AS
                  SELECT
                         ROW_NUMBER() OVER (ORDER BY cst_id) AS customer_key,
                         ci.cst_id AS customer_id,
                         ci.cst_key AS customer_number,
                         ci.cst_firstname AS first_name,
                         ci.cst_lastname AS last_name,
                         la.cntry AS country,
                         ci.cst_material_status AS marital_status,
                         CASE WHEN  ci.cst_material_gndr !='N/A' THEN  ci.cst_material_gndr
                         ELSE COALESCE(ca.gen,'N/A') END AS gender,
                         ca.bdate AS birthdate,
                         ci.cst_create_date
                  FROM silver.crm_cust_info ci
                  LEFT JOIN silver.erp_cust_az12 ca ON ci.cst_key=ca.cid
                  LEFT JOIN silver.erp_loc_a101 la ON ci.cst_key = la.cid;""")
  conn.commit()

In [ ]:
with conn.cursor() as cur:
  cur.execute("""SELECT * FROM gold.dim_customers;""")
  columns=[desc[0]for desc in cur.description]
  rows = cur.fetchall()
df=pd.DataFrame(rows,columns=columns)
display(df)

,customer_key,customer_id,customer_number,first_name,last_name,country,marital_status,gender,birthdate,cst_create_date
0,1,11000.0,AW00011000,Jon,Yang,Australia,Married,Male,1971-10-06,2025-10-06
1,2,11001.0,AW00011001,Eugene,Huang,Australia,Single,Male,1976-05-10,2025-10-06
2,3,11002.0,AW00011002,Ruben,Torres,Australia,Married,Male,1971-02-09,2025-10-06
3,4,11003.0,AW00011003,Christy,Zhu,Australia,Single,Female,1973-08-14,2025-10-06
4,5,11004.0,AW00011004,Elizabeth,Johnson,Australia,Single,Female,1979-08-05,2025-10-06
...,...,...,...,...,...,...,...,...,...,...
18480,18481,29480.0,AW00029480,Nina,Raji,United Kingdom,Single,N/A,1977-05-06,2026-01-25
18481,18482,29481.0,AW00029481,Ivan,Suri,Germany,Single,N/A,1965-07-04,2026-01-25
18482,18483,29482.0,AW00029482,Clayton,Zhang,France,Married,N/A,1964-09-01,2026-01-25
18483,18484,29483.0,AW00029483,Marc,Navarro,France,Married,N/A,1965-06-06,2026-01-27


In [ ]:
#Quality Check of the Gold Table
with conn.cursor() as cur:
  cur.execute("""SELECT distinct gender FROM gold.dim_customers;""")
  columns=[desc[0]for desc in cur.description]
  rows = cur.fetchall()
df=pd.DataFrame(rows,columns=columns)
display(df)


,gender
0,N/A
1,Female
2,Male


In [ ]:
#Create Dimension Products
#WHERE: If End Date is NULL then It is current info of the product!(Filter out all historical data)
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""SELECT
                 pn.prd_id,
                 pn.cat_id,
                 pn.prd_key,
                 pn.prd_nm,
                 pn.prd_cost,
                 pn.prd_line,
                 pn.prd_start_dt,
                 pc.cat,
                 pc.subcat,
                 pc.maintenance,
                 pn.prd_end_dt
                 FROM silver.crm_prd_info pn
                 LEFT JOIN silver.erp_px_cat_g1v2 pc ON pn.cat_id=pc.id
                 WHERE pn.prd_end_dt IS NULL ;""")
  columns=[desc[0]for desc in cur.description]
  rows = cur.fetchall()
df=pd.DataFrame(rows,columns=columns)
display(df)


,prd_id,cat_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,cat,subcat,maintenance,prd_end_dt
0,483,AC_BR,RA_H123,Hitch Rack - 4-Bike,45,other Sales,2013-07-01,Accessories,Bike Racks,Yes,None
1,486,AC_BS,ST_1401,All-Purpose Bike Stand,59,Mountain,2013-07-01,Accessories,Bike Stands,No,None
2,477,AC_BC,WB_H098,Water Bottle - 30 oz.,2,other Sales,2013-07-01,Accessories,Bottles and Cages,No,None
3,479,AC_BC,BC_R205,Road Bottle Cage,3,Road,2013-07-01,Accessories,Bottles and Cages,No,None
4,478,AC_BC,BC_M005,Mountain Bottle Cage,4,Mountain,2013-07-01,Accessories,Bottles and Cages,No,None
...,...,...,...,...,...,...,...,...,...,...,...
290,546,CO_PE,PD_R563,ML Road Pedal,28,Road,2013-07-01,None,None,None,None
291,545,CO_PE,PD_R347,LL Road Pedal,18,Road,2013-07-01,None,None,None,None
292,544,CO_PE,PD_M562,HL Mountain Pedal,36,Mountain,2013-07-01,None,None,None,None
293,543,CO_PE,PD_M340,ML Mountain Pedal,28,Mountain,2013-07-01,None,None,None,None


In [ ]:
#Quailty check#NO DUPLICATE
with conn.cursor() as cur:
  cur.execute("""SELECT prd_key, COUNT(*) FROM
                 (SELECT
                 pn.prd_id,
                 pn.cat_id,
                 pn.prd_key,
                 pn.prd_nm,
                 pn.prd_cost,
                 pn.prd_line,
                 pn.prd_start_dt,
                 pc.cat,
                 pc.subcat,
                 pc.maintenance,
                 pn.prd_end_dt
                 FROM silver.crm_prd_info pn
                 LEFT JOIN silver.erp_px_cat_g1v2 pc ON pn.cat_id=pc.id
                 WHERE pn.prd_end_dt IS NULL)T
                 GROUP BY prd_key
                 HAVING COUNT(*)>1 ;""")
  columns=[desc[0]for desc in cur.description]
  rows = cur.fetchall()
df=pd.DataFrame(rows,columns=columns)
display(df)



,prd_key,count


In [ ]:
#Sorting the columns into logical groups to improve readability and rename all the columns
#Dimension VS Fact?:This table describe the business object products without transactions ->Dimension
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""CREATE VIEW gold.dim_products AS
                 SELECT
                 ROW_NUMBER() OVER (ORDER BY pn.prd_start_dt, pn.prd_key) AS product_key,
                 pn.prd_id AS product_id,
                 pn.prd_key AS product_number,
                 pn.prd_nm AS product_name,
                 pn.cat_id AS category_id,
                 pc.cat AS category,
                 pc.subcat AS subcategory,
                 pc.maintenance,
                 pn.prd_cost AS cost,
                 pn.prd_line AS product_line,
                 pn.prd_start_dt AS start_date,
                 pn.prd_end_dt
                 FROM silver.crm_prd_info pn
                 LEFT JOIN silver.erp_px_cat_g1v2 pc ON pn.cat_id=pc.id
                 WHERE pn.prd_end_dt IS NULL;""")
conn.commit()


In [ ]:
with conn.cursor() as cur:
  cur.execute("""SELECT * FROM gold.dim_products;""")
  columns=[desc[0]for desc in cur.description]
  rows = cur.fetchall()
df=pd.DataFrame(rows,columns=columns)
display(df)

,product_key,product_id,product_number,product_name,category_id,category,subcategory,maintenance,cost,product_line,start_date,prd_end_dt
0,1,210,FR_R92B_58,HL Road Frame - Black- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01,None
1,2,211,FR_R92R_58,HL Road Frame - Red- 58,CO_RF,Components,Road Frames,Yes,0,Road,2003-07-01,None
2,3,348,BK_M82B_38,Mountain-100 Black- 38,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01,None
3,4,349,BK_M82B_42,Mountain-100 Black- 42,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01,None
4,5,350,BK_M82B_44,Mountain-100 Black- 44,BI_MB,Bikes,Mountain Bikes,Yes,1898,Mountain,2011-07-01,None
...,...,...,...,...,...,...,...,...,...,...,...,...
290,291,530,TT_T092,Touring Tire Tube,AC_TT,Accessories,Tires and Tubes,Yes,2,Touring,2013-07-01,None
291,292,473,VE_C304_L,Classic Vest- L,CL_VE,Clothing,Vests,No,24,other Sales,2013-07-01,None
292,293,472,VE_C304_M,Classic Vest- M,CL_VE,Clothing,Vests,No,24,other Sales,2013-07-01,None
293,294,471,VE_C304_S,Classic Vest- S,CL_VE,Clothing,Vests,No,24,other Sales,2013-07-01,None


# CREATE FACT SALES


In [ ]:
conn.rollback()
with conn.cursor() as cur:
  cur.execute("""SELECT
                 sd.sls_ord_num,
                 sd.sls_prd_key,
                 sd.sls_cust_id,
                 sd.sls_order_dt,
                 sd.sls_ship_dt,
                 sd.sls_due_dt,
                 sd.sls_sales,
                 sd.sls_quantity,
                 sd.sls_price
                 FROM silver.crm_sales_details sd;""")
  columns=[desc[0]for desc in cur.description]
  rows=cur.fetchall()
df=pd.DataFrame(rows,columns=columns)
display(df)
#We can see a lots of Fact table's details such as keys(ids),dates, and measures, therefore this one is fact
#Fact is connecting multiple dimensions
#sls_prd_key, sls_cust_id come from souce system. We are going to replace those two info with the surrogate keys so that we can regenerate
#Building Fact:Use the dimension's surrogate keys instead of IDs to easily connect facts with dimensions 'data lookup'
#We need the circuit key that we can generated in our own in this data warehouse
#Dimension keys  pr.product_key, cu.customer_key: you have to put the surrogate keys from the dimensins into the fact
#Rename all the columns and arrange the columns
with conn.cursor() as cur:
  cur.execute("""SELECT
                 sd.sls_ord_num AS order_number,
                 pr.product_key,
                 cu.customer_key,
                 sd.sls_order_dt AS order_date,
                 sd.sls_ship_dt AS shipping_date,
                 sd.sls_due_dt AS due_date,
                 sd.sls_sales AS sales_amount,
                 sd.sls_quantity AS quantity,
                 sd.sls_price
                 FROM silver.crm_sales_details sd
                 LEFT JOIN gold.dim_products pr ON sd.sls_prd_key = pr.product_number
                 LEFT JOIN gold.dim_customers cu ON sd.sls_cust_id = cu.customer_id ;""")
  columns=[desc[0]for desc in cur.description]
  rows=cur.fetchall()
df=pd.DataFrame(rows,columns=columns)
display(df)


,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
0,SO67993,PK-7098,17084,2013-10-08,2013-10-15,2013-10-20,2,2,None
1,SO68588,PK-7098,11318,2013-10-17,2013-10-24,2013-10-29,2,2,None
2,SO68889,TT-R982,14086,2013-10-22,2013-10-29,2013-11-03,4,4,None
3,SO61561,CL-9009,18380,2013-07-05,2013-07-12,2013-07-17,8,8,None
4,SO68551,BC-R205,22691,2013-10-17,2013-10-24,2013-10-29,9,9,None
5,SO61548,CA-1098,12386,2013-07-05,2013-07-12,2013-07-17,9,9,None
6,SO61570,CA-1098,17809,2013-07-05,2013-07-12,2013-07-17,9,9,None
7,SO61636,BC-M005,12914,2013-07-06,2013-07-13,2013-07-18,10,10,None
8,SO67551,BC-M005,15150,2013-10-01,2013-10-08,2013-10-13,10,10,None
9,SO67707,TT-M928,26046,2013-10-04,2013-10-11,2013-10-16,10,5,None


,order_number,product_key,customer_key,order_date,shipping_date,due_date,sales_amount,quantity,sls_price
0,SO68588,None,319,2013-10-17,2013-10-24,2013-10-29,2,2,None
1,SO58326,None,1046,2013-05-20,2013-05-27,2013-06-01,22,22,None
2,SO61548,None,1387,2013-07-05,2013-07-12,2013-07-17,100,50,None
3,SO61548,None,1387,2013-07-05,2013-07-12,2013-07-17,9,9,None
4,SO61636,None,1915,2013-07-06,2013-07-13,2013-07-18,10,10,None
5,SO58335,None,2327,2013-05-20,2013-05-27,2013-06-01,70,35,None
6,SO68126,None,2605,2013-10-10,2013-10-17,2013-10-22,50,50,None
7,SO68889,None,3087,2013-10-22,2013-10-29,2013-11-03,4,4,None
8,SO61879,None,3456,2013-07-10,2013-07-17,2013-07-22,100,50,None
9,SO67551,None,4151,2013-10-01,2013-10-08,2013-10-13,10,10,None


In [ ]:
#Create a view
with conn.cursor() as cur:
  cur.execute("""CREATE VIEW gold.fact_sales AS
                 SELECT
                 sd.sls_ord_num AS order_number,
                 pr.product_key,
                 cu.customer_key,
                 sd.sls_order_dt AS order_date,
                 sd.sls_ship_dt AS shipping_date,
                 sd.sls_due_dt AS due_date,
                 sd.sls_sales AS sales_amount,
                 sd.sls_quantity AS quantity,
                 sd.sls_price
                 FROM silver.crm_sales_details sd
                 LEFT JOIN gold.dim_products pr ON sd.sls_prd_key = pr.product_number
                 LEFT JOIN gold.dim_customers cu ON sd.sls_cust_id = cu.customer_id ;""")
conn.commit()
with conn.cursor() as cur:
  cur.execute("""SELECT * FROM gold.fact_sales;""" )
  columns=[desc[0]for desc in cur.description]
  rows = cur.fetchall()
df=pd.DataFrame(rows,columns=columns)
display(df)

,order_number,product_key,customer_key,order_date,shipping_date,due_date,sales_amount,quantity,sls_price
0,SO68588,None,319,2013-10-17,2013-10-24,2013-10-29,2,2,None
1,SO58326,None,1046,2013-05-20,2013-05-27,2013-06-01,22,22,None
2,SO61548,None,1387,2013-07-05,2013-07-12,2013-07-17,100,50,None
3,SO61548,None,1387,2013-07-05,2013-07-12,2013-07-17,9,9,None
4,SO61636,None,1915,2013-07-06,2013-07-13,2013-07-18,10,10,None
5,SO58335,None,2327,2013-05-20,2013-05-27,2013-06-01,70,35,None
6,SO68126,None,2605,2013-10-10,2013-10-17,2013-10-22,50,50,None
7,SO68889,None,3087,2013-10-22,2013-10-29,2013-11-03,4,4,None
8,SO61879,None,3456,2013-07-10,2013-07-17,2013-07-22,100,50,None
9,SO67551,None,4151,2013-10-01,2013-10-08,2013-10-13,10,10,None


In [ ]:
#Quality Check of the Gold Table
#---Foreign Key Integrity (Dimensions)
with conn.cursor() as cur:
  cur.execute("""SELECT *
                 FROM gold.fact_sales f
                 LEFT JOIN gold.dim_customers c ON c.customer_key=f.customer_key
                 LEFT JOIN gold.dim_products p ON p.product_key = f.product_key
                 WHERE c.customer_key IS NULL;""")
  columns=[desc[0]for desc in cur.description]
  rows = cur.fetchall()
df=pd.DataFrame(rows,columns=columns)
display(df)


,order_number,product_key,customer_key,order_date,shipping_date,due_date,sales_amount,quantity,sls_price,customer_key,...,product_number,product_name,category_id,category,subcategory,maintenance,cost,product_line,start_date,prd_end_dt
